# EDA — Home Credit Default Risk
### FinShield AI · Scoring Crédit

Analyse exploratoire des données de demande de pret. On cherche a comprendre la structure du dataset, identifier les features qui ont un vrai pouvoir predictif, et detecter les problemes latents avant de toucher au pipeline ML.

> **Important** : toute l'analyse est faite sur X_train uniquement apres le split. On ne regarde jamais le jeu de test pendant l'EDA — meme les distributions.

---

## 0. Setup

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn scipy statsmodels pyarrow -q
print('ok')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from scipy.stats import mannwhitneyu, chi2_contingency, spearmanr
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams['figure.dpi']        = 110
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size']         = 10

C1, C2, C3 = '#1B3F72', '#E74C3C', '#2ECC71'
SEED = 42

os.makedirs('/content/docs', exist_ok=True)
os.makedirs('/content/processed', exist_ok=True)

## 1. Chargement + split immediat

Le split se fait ici, avant toute analyse. Comme ca on est sur de ne jamais contaminer nos conclusions avec des infos du jeu de test.

In [ ]:
from google.colab import drive
drive.mount('/drive')

df_full = pd.read_csv('/drive/MyDrive/finshield-ai/data/raw/application_train.csv')
print(f'Dataset complet : {df_full.shape}')

y_full = df_full['TARGET']
X_full = df_full.drop(columns=['TARGET', 'SK_ID_CURR'])

# split stratifié — on ne touche plus a X_test apres ca
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full,
    test_size=0.20, stratify=y_full, random_state=SEED
)

# on recolle y_train pour l'EDA
train = X_train.copy()
train['TARGET'] = y_train.values

print(f'X_train : {X_train.shape}  —  défaut : {y_train.mean():.2%}')
print(f'X_test  : {X_test.shape}   —  défaut : {y_test.mean():.2%}')
print('X_test gelé — on ny touche plus jusqu au notebook modeling')

## 2. Validaton de base

Avant de regarder quoi que ce soit, on verife l'integrite des données — doublons, types, valeurs impossibles.

In [ ]:
# doublons — classique sur les datasets kaggle
dupes_exact = train.duplicated().sum()

print(f'Doublons exacts          : {dupes_exact}')
print(f'Colonnes numériques      : {train.select_dtypes(include=np.number).shape[1]}')
print(f'Colonnes catégorielles   : {train.select_dtypes(include="object").shape[1]}')
print(f'Mémoire                  : {train.memory_usage(deep=True).sum()/1e6:.1f} MB')

# valeurs impossibles connues dans ce dataset
n_days_aberrant = (train['DAYS_EMPLOYED'] == 365243).sum()
print(f'\nDAYS_EMPLOYED = 365243   : {n_days_aberrant} lignes ({n_days_aberrant/len(train):.1%})')
# 365243 jours = ~1000 ans — clairement du codage pour "sans emploi"

## 3. Variable cible — distribution et desequilibre

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# comptage
counts = train['TARGET'].value_counts()
bars = axes[0].bar(
    ['Remboursé (0)', 'Défaut (1)'],
    counts.values,
    color=[C1, C2], edgecolor='white', linewidth=1.5, width=0.5
)
for bar, val in zip(bars, counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 1500,
        f'{val:,}\n({val/len(train):.1%})',
        ha='center', fontweight='bold', fontsize=9
    )
axes[0].set_title('Distribution TARGET — X_train', fontweight='bold')
axes[0].set_ylabel('Nombre de clients')
axes[0].set_ylim(0, counts.max() * 1.2)

# pie
axes[1].pie(
    counts.values,
    labels=['Remboursé', 'Défaut'],
    colors=[C1, C2],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 10}
)
axes[1].set_title('Proportion TARGET', fontweight='bold')

plt.suptitle('Desequilibre de classes — 1 défaut pour ~11 remboursés',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/docs/target_dist.png', bbox_inches='tight', dpi=150)
plt.show()

ratio = counts[0] / counts[1]
print(f'Ratio deséquilibre : 1 défaut pour {ratio:.0f} non-défauts')
print(f'→ scale_pos_weight XGBoost = {ratio:.2f}')

## 4. Analyse des valeurs manquantes + test MCAR vs MNAR

Le point cle ici c'est pas juste de compter les NaN — c'est de comprendre *pourquoi* ils manquent. Si une valeur manque a cause de sa propre valeur (ex : mauvais score credit donc pas fourni), l'imputation par mediane introduit un biais systematique.

In [ ]:
# calcul missing sur X_train uniquement
missing = pd.DataFrame({
    'count' : X_train.isnull().sum(),
    'pct'   : X_train.isnull().mean() * 100
}).query('count > 0').sort_values('pct', ascending=False)

print(f'Colonnes avec NaN : {len(missing)} / {X_train.shape[1]}')
print(f'Colonnes > 50% NaN : {(missing["pct"] > 50).sum()}')

# visualisation top 30
top30 = missing.head(30)
fig, ax = plt.subplots(figsize=(11, 8))
colors_bar = [C2 if v > 50 else C1 for v in top30['pct']]
ax.barh(top30.index[::-1], top30['pct'][::-1], color=colors_bar[::-1], alpha=0.85)
ax.axvline(x=50, color=C2, linestyle='--', linewidth=1.5, label='Seuil 50%')
ax.set_xlabel('% valeurs manquantes')
ax.set_title('Top 30 colonnes — valeurs manquantes (X_train)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('/content/docs/missing_values.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# test MCAR vs MNAR — si le flag NaN est corrélé avec TARGET → MNAR
# c'est le cas le plus dangereux pour l'imputation
mnar_results = []
missing_cols = missing.index.tolist()

for col in missing_cols:
    flag = X_train[col].isnull().astype(int)
    corr = flag.corr(y_train)
    mnar_results.append({'feature': col, 'missing_pct': missing.loc[col, 'pct'], 'corr_with_target': corr})

mnar_df = pd.DataFrame(mnar_results).sort_values('corr_with_target', key=abs, ascending=False)

print('Features dont le NaN est corrélé avec TARGET (MNAR probable) :')
print('Ces features nécessitent un FLAG_MISSING en plus de l imputation\n')
mnar_important = mnar_df[mnar_df['corr_with_target'].abs() > 0.05]
print(mnar_important.to_string(index=False, float_format='{:.4f}'.format))

# visualisation
fig, ax = plt.subplots(figsize=(11, 5))
top_mnar = mnar_df.head(20)
colors_mnar = [C2 if abs(v) > 0.05 else C1 for v in top_mnar['corr_with_target']]
ax.barh(top_mnar['feature'][::-1], top_mnar['corr_with_target'][::-1],
        color=colors_mnar[::-1], alpha=0.85)
ax.axvline(x=0.05,  color=C2, linestyle='--', linewidth=1, alpha=0.7)
ax.axvline(x=-0.05, color=C2, linestyle='--', linewidth=1, alpha=0.7, label='Seuil |0.05|')
ax.set_xlabel('Corrélation (flag NaN ↔ TARGET)')
ax.set_title('Test MCAR vs MNAR — Corrélation flag manquant avec TARGET', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('/content/docs/mnar_test.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Outliers — detection IQR systematique

On utilise le critere 3×IQR plutot que 1.5×IQR pour etre plus conservateur — en finance les distributions ont des queues lourdes (heavy tails) et 1.5×IQR est trop agressif.

In [ ]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

outlier_report = []
for col in num_cols:
    s = X_train[col].dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 3*IQR, Q3 + 3*IQR
    n_out = ((s < lower) | (s > upper)).sum()
    if n_out > 0:
        outlier_report.append({
            'feature'   : col,
            'n_outliers': n_out,
            'pct'       : n_out / len(s) * 100,
            'lower'     : lower,
            'upper'     : upper
        })

outlier_df = pd.DataFrame(outlier_report).sort_values('pct', ascending=False)

print(f'Features avec outliers (3×IQR) : {len(outlier_df)}')
print('\nTop 15 features les plus affectées :')
print(outlier_df.head(15)[['feature','n_outliers','pct']].to_string(index=False, float_format='{:.2f}'.format))

# visualisation — focus sur les 3 features clés
focus_cols = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, focus_cols):
    data = X_train[col].dropna()
    # clip a 99e percentile pour la visu
    clip_val = data.quantile(0.99)
    ax.hist(data.clip(upper=clip_val), bins=60, color=C1, alpha=0.8, edgecolor='none')
    ax.axvline(data.median(), color=C2, linewidth=2, linestyle='--',
               label=f'Médiane : {data.median():,.0f}')
    ax.set_title(col, fontweight='bold', fontsize=9)
    ax.legend(fontsize=7)
    ax.set_ylabel('Count')

plt.suptitle('Distribution des montants — queue droite très longue (heavy tail)',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/docs/outliers_amounts.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Corrélations — Spearman pas Pearson

Pearson suppose des distributions normales et des relations lineaires — aucune de ces conditions n'est remplie ici. Spearman mesure les relations monotones non-lineaires, beaucoup plus adapte aux données financieres.

In [ ]:
# calcul Pearson vs Spearman sur X_train — la diff revele les relations non-lineaires
key_num = [
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'AMT_INCOME_TOTAL',
    'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'REGION_POPULATION_RELATIVE', 'DAYS_REGISTRATION',
    'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'CNT_CHILDREN'
]
key_num = [c for c in key_num if c in X_train.columns]

corr_results = []
for col in key_num:
    valid = X_train[[col]].join(y_train).dropna()
    p_corr = valid[col].corr(valid['TARGET'], method='pearson')
    s_corr = valid[col].corr(valid['TARGET'], method='spearman')
    corr_results.append({'feature': col, 'pearson': p_corr, 'spearman': s_corr,
                         'delta': abs(s_corr - p_corr)})

corr_df = pd.DataFrame(corr_results).sort_values('spearman', key=abs, ascending=False)

# visualisation comparaison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(corr_df))
w = 0.35
b1 = ax.bar(x - w/2, corr_df['pearson'],  w, label='Pearson',  color=C1, alpha=0.8)
b2 = ax.bar(x + w/2, corr_df['spearman'], w, label='Spearman', color=C2, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(corr_df['feature'], rotation=40, ha='right', fontsize=8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Corrélation avec TARGET')
ax.set_title('Pearson vs Spearman — les différences révèlent des relations non-linéaires',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('/content/docs/corr_pearson_vs_spearman.png', bbox_inches='tight', dpi=150)
plt.show()

print('Top features par corrélation Spearman avec TARGET :')
print(corr_df[['feature','pearson','spearman','delta']].to_string(index=False, float_format='{:+.4f}'.format))

## 7. Tests statistiques — Mann-Whitney U

La corrélation dit si une relation existe. Le test Mann-Whitney dit si cette difference est statistiquement significative. Sur 250k lignes presque tout sera significatif — donc on calcule aussi l'effect size (rank-biserial) pour savoir si c'est *pratiquement* important.

In [ ]:
mw_results = []
for col in key_num:
    g0 = X_train.loc[y_train==0, col].dropna()
    g1 = X_train.loc[y_train==1, col].dropna()
    if len(g0) > 10 and len(g1) > 10:
        stat, pval = mannwhitneyu(g0, g1, alternative='two-sided')
        # rank-biserial effect size — 0=aucun effet, 1=effet parfait
        effect = 1 - (2*stat) / (len(g0)*len(g1))
        mw_results.append({
            'feature'    : col,
            'p_value'    : pval,
            'effect_size': abs(effect),
            'significant': pval < 0.001
        })

mw_df = pd.DataFrame(mw_results).sort_values('effect_size', ascending=False)

# visualisation effect size
fig, ax = plt.subplots(figsize=(11, 5))
colors_eff = [C2 if v > 0.1 else C1 for v in mw_df['effect_size']]
ax.barh(mw_df['feature'][::-1], mw_df['effect_size'][::-1],
        color=colors_eff[::-1], alpha=0.85)
ax.axvline(0.1, color=C2, linestyle='--', linewidth=1.5, label='Seuil effect size 0.10')
ax.set_xlabel('Effect size (rank-biserial)')
ax.set_title('Mann-Whitney U — Effect size par feature (X_train)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('/content/docs/mannwhitney_effect.png', bbox_inches='tight', dpi=150)
plt.show()

print('\nTop features par effect size (Mann-Whitney) :')
print(mw_df.to_string(index=False, float_format='{:.4f}'.format))

## 8. Weight of Evidence / Information Value

Standard de l'industrie bancaire. L'IV classe les features par pouvoir predictif :

```
< 0.02  → inutile
0.02–0.10 → faible
0.10–0.30 → moyen
0.30–0.50 → fort
> 0.50  → suspicion de leakage
```

In [ ]:
def compute_iv(series, target, bins=10):
    """Calcule l'Information Value d'une feature continue."""
    df_tmp = pd.DataFrame({'x': series, 'y': target}).dropna()
    try:
        df_tmp['bin'] = pd.qcut(df_tmp['x'], q=bins, duplicates='drop')
    except:
        return np.nan
    grouped = df_tmp.groupby('bin')['y'].agg(['sum','count'])
    grouped.columns = ['events', 'total']
    grouped['non_events'] = grouped['total'] - grouped['events']
    total_events     = grouped['events'].sum()
    total_non_events = grouped['non_events'].sum()
    grouped['dist_events']     = grouped['events']     / (total_events     + 1e-10)
    grouped['dist_non_events'] = grouped['non_events'] / (total_non_events + 1e-10)
    grouped['woe'] = np.log(
        (grouped['dist_events'] + 1e-10) / (grouped['dist_non_events'] + 1e-10)
    )
    grouped['iv_contrib'] = (grouped['dist_events'] - grouped['dist_non_events']) * grouped['woe']
    return grouped['iv_contrib'].sum()

iv_results = []
for col in key_num:
    iv = compute_iv(X_train[col], y_train)
    if not np.isnan(iv):
        if iv < 0.02:       power = 'Inutile'
        elif iv < 0.10:     power = 'Faible'
        elif iv < 0.30:     power = 'Moyen'
        elif iv < 0.50:     power = 'Fort'
        else:               power = '⚠️ Suspect'
        iv_results.append({'feature': col, 'IV': iv, 'Pouvoir': power})

iv_df = pd.DataFrame(iv_results).sort_values('IV', ascending=False)

# visualisation
fig, ax = plt.subplots(figsize=(11, 5))
color_map = {'Inutile':'#BDC3C7','Faible':'#85C1E9','Moyen':C1,'Fort':C2,'⚠️ Suspect':'#F39C12'}
colors_iv = [color_map.get(p, C1) for p in iv_df['Pouvoir']]
ax.barh(iv_df['feature'][::-1], iv_df['IV'][::-1], color=colors_iv[::-1], alpha=0.9)
ax.axvline(0.10, color='grey',  linestyle=':', linewidth=1.5, label='IV = 0.10 (faible→moyen)')
ax.axvline(0.30, color=C2,      linestyle='--', linewidth=1.5, label='IV = 0.30 (moyen→fort)')
ax.set_xlabel('Information Value')
ax.set_title('Information Value par feature — standard industrie bancaire', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('/content/docs/information_value.png', bbox_inches='tight', dpi=150)
plt.show()

print(iv_df.to_string(index=False, float_format='{:.4f}'.format))

## 9. Multicolinearite — VIF

On verifie que les features qu'on va garder ne sont pas trop correlees entre elles. Un VIF > 5 indique un probleme — la feature n'apporte pas d'information independante et gonfle la variance du modele.

In [ ]:
from sklearn.impute import SimpleImputer

# VIF sur les features clés numériques — apres imputation
vif_cols = [c for c in key_num if X_train[c].isnull().mean() < 0.6]
X_vif    = X_train[vif_cols].copy()

imp = SimpleImputer(strategy='median')
X_vif_imp = pd.DataFrame(imp.fit_transform(X_vif), columns=vif_cols)

vif_data = pd.DataFrame()
vif_data['feature'] = vif_cols
vif_data['VIF']     = [
    variance_inflation_factor(X_vif_imp.values, i)
    for i in range(X_vif_imp.shape[1])
]
vif_data = vif_data.sort_values('VIF', ascending=False)

# visualisation
fig, ax = plt.subplots(figsize=(11, 5))
colors_vif = [C2 if v > 5 else C1 for v in vif_data['VIF']]
ax.barh(vif_data['feature'][::-1], vif_data['VIF'][::-1],
        color=colors_vif[::-1], alpha=0.85)
ax.axvline(5,  color=C2,    linestyle='--', linewidth=1.5, label='VIF = 5  (seuil modéré)')
ax.axvline(10, color='#900', linestyle='--', linewidth=1.5, label='VIF = 10 (seuil critique)')
ax.set_xlabel('Variance Inflation Factor')
ax.set_title('Multicolinéarité — VIF par feature', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('/content/docs/vif_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

problematic = vif_data[vif_data['VIF'] > 5]
if len(problematic) > 0:
    print('Features avec VIF > 5 — a traiter :')
    print(problematic.to_string(index=False, float_format='{:.2f}'.format))
else:
    print('Aucune multicolinearite critique detectée')

## 10. Features categorelles — V de Cramer

L'equivalent du IV pour les categorelles. Le V de Cramer mesure la force de l'association entre une variable categorielle et la cible, de 0 (aucune association) a 1 (association parfaite).

In [ ]:
def cramers_v(x, y):
    cm = pd.crosstab(x, y)
    chi2, pval, dof, _ = chi2_contingency(cm)
    n = cm.values.sum()
    v = np.sqrt(chi2 / (n * (min(cm.shape) - 1)))
    return v, pval

cramer_results = []
for col in cat_cols:
    v, pval = cramers_v(X_train[col].fillna('MISSING'), y_train)
    cramer_results.append({'feature': col, 'cramers_v': v, 'p_value': pval})

cramer_df = pd.DataFrame(cramer_results).sort_values('cramers_v', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors_cr = [C2 if v > 0.05 else '#BDC3C7' for v in cramer_df['cramers_v']]
ax.barh(cramer_df['feature'][::-1], cramer_df['cramers_v'][::-1],
        color=colors_cr[::-1], alpha=0.85)
ax.axvline(0.05, color=C2, linestyle='--', linewidth=1.5, label='V = 0.05 (seuil minimal)')
ax.set_xlabel('Cramér V')
ax.set_title('Association features catégorielles ↔ TARGET', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('/content/docs/cramers_v.png', bbox_inches='tight', dpi=150)
plt.show()

print(cramer_df.to_string(index=False, float_format='{:.4f}'.format))

## 11. Analyse EXT_SOURCE — les features les plus predicitves

Ces trois colonnes sont des scores de credit externes. Ce sont de loin les meilleurs predicteurs du modele. On les analyse en detail car leur strategie d'imputation a un impact direct sur l'AUC.

In [ ]:
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for i, col in enumerate(ext_cols):
    # distribution par TARGET
    for tgt, color, label in [(0, C1, 'Remboursé'), (1, C2, 'Défaut')]:
        data = X_train.loc[y_train==tgt, col].dropna()
        axes[0, i].hist(data, bins=50, alpha=0.6, color=color, label=label, density=True)
    axes[0, i].set_title(col, fontweight='bold')
    axes[0, i].legend(fontsize=8)
    axes[0, i].set_xlabel('Score')

    # boxplot
    data_0 = X_train.loc[y_train==0, col].dropna()
    data_1 = X_train.loc[y_train==1, col].dropna()
    bp = axes[1, i].boxplot(
        [data_0, data_1],
        labels=['Remboursé', 'Défaut'],
        patch_artist=True,
        notch=True
    )
    for patch, color in zip(bp['boxes'], [C1, C2]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    for median in bp['medians']:
        median.set_color('white')
        median.set_linewidth(2)
    axes[1, i].set_title(f'{col} — médiane par classe', fontweight='bold', fontsize=9)

plt.suptitle('EXT_SOURCE 1/2/3 — Plus le score est bas, plus le risque est élevé',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/docs/ext_source_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

# corrélation entre elles — si trop haute on n'a pas besoin des 3
ext_corr = X_train[ext_cols].corr(method='spearman')
print('Corrélation Spearman entre EXT_SOURCE :')
print(ext_corr.round(4))

## 12. Taux de defaut par feature — analyse metier

In [ ]:
# age — feature metier importante
train_clean = train.copy()
train_clean['DAYS_EMPLOYED'] = train_clean['DAYS_EMPLOYED'].replace(365243, np.nan)
train_clean['AGE_YEARS']     = -train_clean['DAYS_BIRTH'] / 365
train_clean['AGE_BIN']       = pd.cut(
    train_clean['AGE_YEARS'],
    bins=[20, 25, 30, 35, 40, 45, 50, 55, 60, 70],
    labels=['20-25','25-30','30-35','35-40','40-45','45-50','50-55','55-60','60+']
)

age_stats = train_clean.groupby('AGE_BIN')['TARGET'].agg(['mean','count']).reset_index()
global_rate = train['TARGET'].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors_age = [C2 if v > global_rate else C1 for v in age_stats['mean']]
axes[0].bar(age_stats['AGE_BIN'], age_stats['mean'],
            color=colors_age, edgecolor='white', alpha=0.9)
axes[0].axhline(global_rate, color='orange', linestyle='--', linewidth=2,
                label=f'Taux global : {global_rate:.2%}')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.1%}'))
axes[0].set_title('Taux de défaut par tranche d age', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend()

axes[1].bar(age_stats['AGE_BIN'], age_stats['count'], color=C1, edgecolor='white', alpha=0.85)
axes[1].set_title('Volume par tranche d age', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Jeunes clients (20-30 ans) — taux de défaut significativement plus élevé',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/docs/age_default_rate.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# taux de defaut par features categorielles cles
cat_key = ['CODE_GENDER', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS']
cat_key = [c for c in cat_key if c in X_train.columns]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, col in enumerate(cat_key):
    tmp = pd.DataFrame({'x': X_train[col].values, 'target': y_train.values})
    rate = tmp.groupby('x')['target'].mean().sort_values(ascending=False)
    colors_cat = [C2 if v > global_rate else C1 for v in rate.values]
    axes[i].bar(range(len(rate)), rate.values, color=colors_cat, edgecolor='white', alpha=0.9)
    axes[i].set_xticks(range(len(rate)))
    axes[i].set_xticklabels(rate.index, rotation=30, ha='right', fontsize=8)
    axes[i].axhline(global_rate, color='orange', linestyle='--', linewidth=1.5)
    axes[i].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.1%}'))
    axes[i].set_title(col, fontweight='bold')

plt.suptitle('Taux de défaut par feature catégorielle (X_train)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/docs/cat_default_rates.png', bbox_inches='tight', dpi=150)
plt.show()

## 13. Matrice de correlation — heatmap

In [ ]:
# matrice Spearman sur les features cles + TARGET
corr_cols = [c for c in key_num if X_train[c].isnull().mean() < 0.55] + ['TARGET']
corr_cols = [c for c in corr_cols if c in train.columns]

corr_matrix = train[corr_cols].corr(method='spearman')

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.4,
    annot_kws={'size': 7},
    ax=ax
)
ax.set_title('Matrice de corrélation Spearman — features clés + TARGET',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('/content/docs/corr_matrix_spearman.png', bbox_inches='tight', dpi=150)
plt.show()

## 14. Synthese EDA

Ce qu'on a appris sur X_train — les conclusions qui vont guider le feature engineering dans le notebook modeling.

In [ ]:
print('=' * 65)
print('SYNTHESE EDA v2 — HOME CREDIT DEFAULT RISK (X_train uniquement)')
print('=' * 65)

print(f'''
Dataset
  Lignes train   : {len(X_train):,}
  Taux de defaut : {y_train.mean():.2%}
  Scale pos wt   : {(y_train==0).sum()/(y_train==1).sum():.2f}

Missing data
  Colonnes avec NaN    : {len(missing)}
  Colonnes > 50% NaN   : {(missing["pct"]>50).sum()}
  Features MNAR (⚠️)   : {len(mnar_df[mnar_df["corr_with_target"].abs() > 0.05])}
  → Ces features nécessitent un FLAG_MISSING en plus de l imputation

Features les plus predictives (IV + Spearman + Mann-Whitney)
  1. EXT_SOURCE_2     IV={compute_iv(X_train["EXT_SOURCE_2"], y_train):.3f}
  2. EXT_SOURCE_3     IV={compute_iv(X_train["EXT_SOURCE_3"], y_train):.3f}
  3. EXT_SOURCE_1     IV={compute_iv(X_train["EXT_SOURCE_1"], y_train):.3f}
  4. DAYS_BIRTH       (age — jeunes + risques)
  5. DAYS_EMPLOYED    (ancienneté emploi)

Features categorielles
  Meilleures (V de Cramér) : {cramer_df.head(3)["feature"].tolist()}

Outliers
  DAYS_EMPLOYED = 365243 → remplacer par NaN
  AMT_INCOME_TOTAL    → heavy tail, cap recommandé

Actions pour le pipeline ML
  1. Créer FLAG_MISSING pour features MNAR
  2. Imputation médiane (num) + mode (cat) dans sklearn Pipeline
  3. Corrélations EXT_SOURCE élevées → conserver MEAN + chaque source séparément
  4. VIF > 5 sur certaines features → regularisation L1/L2 suffisante
  5. StratifiedKFold k=5 obligatoire (8% de défauts)
  6. scale_pos_weight = {(y_train==0).sum()/(y_train==1).sum():.2f}
''')
print('=' * 65)

In [ ]:
# sauvegarde X_train propre pour le notebook modeling
# SANS feature engineering — ca sera fait dans la Pipeline
X_train.to_parquet('/content/processed/X_train_raw.parquet', index=False)
X_test.to_parquet('/content/processed/X_test_raw.parquet',  index=False)
y_train.to_frame().to_parquet('/content/processed/y_train.parquet', index=False)
y_test.to_frame().to_parquet('/content/processed/y_test.parquet',   index=False)

# sauvegarde sur Drive
import shutil, os
os.makedirs('/drive/MyDrive/finshield-ai/data/processed', exist_ok=True)
for f in ['X_train_raw','X_test_raw','y_train','y_test']:
    shutil.copy(f'/content/processed/{f}.parquet',
                f'/drive/MyDrive/finshield-ai/data/processed/{f}.parquet')

print('Splits sauvegardés sur Drive — prêts pour le notebook modeling')